In [20]:
!uv pip install numpy

Using Python 3.12.3 environment at: /home/jovyan/.jupyter
Checked 1 package in 38ms


In [21]:
!uv pip install scipy --python /opt/venv/bin/python --target ~/.local/lib/python3.12/site-packages

Using CPython 3.12.3 interpreter at: /opt/venv/bin/python
Checked 1 package in 37ms


In [22]:
import sys
sys.path.insert(0, '/home/jovyan/.local/lib/python3.12/site-packages')

import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

import wnt_model_physics as phys
import wnt_reference_solver as ref
import wnt_pinn_trainer as tr

print('device:', tr.DEVICE)
print('trainer using stub physics:', tr.USING_STUB, '(should be False)')

device: cuda
trainer using stub physics: False (should be False)


In [23]:
def reference_dict(model):
    t, U, P = ref.solve_reference(model)
    BcatTcf = ref.bcat_tcf(U, P)
    i = phys  # reuse index names
    d = dict(t=t,
             H5=U[:, i.iH5], H13=U[:, i.iH13], M=U[:, i.iM], Mi=U[:, i.iMi],
             Ca=U[:, i.iCa], Ci=U[:, i.iCi], Ba=U[:, i.iBa], P=U[:, i.iP],
             Bp=U[:, i.iBp], BcatTcf=BcatTcf, V=U[:, i.iV], Di=U[:, i.iDi],
             Db=U[:, i.iDb], Da=U[:, i.iDa], X=U[:, i.iX], Nr=U[:, i.iNr],
             R=U[:, i.iR], U=U)
    return d

ref_const = reference_dict('const')
print('const reference:', ref_const['U'].shape, 'finite:', np.isfinite(ref_const['U']).all())
# dyn is slower (~minutes) — uncomment when ready:
ref_dyn = reference_dict('dyn')
print('dyn reference:', ref_dyn['U'].shape, 'finite:', np.isfinite(ref_dyn['U']).all())

const reference: (800, 27) finite: True


/home/jovyan/.local/lib/python3.12/site-packages/scipy/integrate/_ivp/common.py:356: RuntimeWarning: overflow encountered in multiply
  new_factor = NUM_JAC_FACTOR_INCREASE * factor[ind]
/home/jovyan/.local/lib/python3.12/site-packages/scipy/integrate/_ivp/common.py:378: RuntimeWarning: overflow encountered in multiply
  factor[max_diff < NUM_JAC_DIFF_SMALL * scale] *= NUM_JAC_FACTOR_INCREASE


dyn reference: (5000, 27) finite: True


In [24]:
def residual_at_reference(model, refd, n_interior=400):
    P = phys.build_params(model)
    t = refd['t']; U = refd['U']
    # central finite-difference derivative (interior points only)
    dU = np.gradient(U, t, axis=0)
    sl = slice(2, len(t) - 2)            # drop noisy endpoints
    tt = torch.tensor(t[sl]).reshape(-1, 1)
    zz = torch.tensor(U[sl])
    dz = torch.tensor(dU[sl])
    re, ri = phys.residuals(tt, zz, dz, P)
    res = torch.cat([re, ri], 1).abs()
    print(f'[{model}] max|res|={res.max().item():.2e}  '
          f'median|res|={res.median().item():.2e}')
    # report the worst few equations (these are the stiff/fast ones; FD-limited)
    per_eq = res.mean(0)
    worst = torch.argsort(per_eq, descending=True)[:5]
    print('   worst eqs (residual index):',
          [(int(k), f'{per_eq[k].item():.1e}') for k in worst])

residual_at_reference('const', ref_const)
# residual_at_reference('dyn', ref_dyn)

[const] max|res|=7.54e-01  median|res|=1.73e-08
   worst eqs (residual index): [(17, '7.2e-01'), (0, '3.1e-01'), (3, '2.6e-01'), (1, '1.1e-01'), (9, '1.8e-04')]


In [ ]:
# ---- SMOKE config (fast; proves the loss drops) ----
SMOKE = dict(win_len=1000.0, iters=300, lr=2e-3, n_col=1024,
             n_out=120, lbfgs_iters=20, max_windows=3)

# ---- FULL config (GPU; tune these) ----
FULL = dict(win_len=500.0, iters=4000, lr=2e-3, n_col=4096,
            n_out=300, lbfgs_iters=80, max_windows=None)

cfg = FULL   # <-- set to FULL on the GPU VM

# Per-window checkpoints: resume automatically if the job dies.
# Set to None to disable (e.g. for SMOKE runs).
CKPT_DIR = "pinn_checkpoints"

pinn_const = tr.solve_pinn('const', gamma1=1.0, checkpoint_dir=f"{CKPT_DIR}/const", **cfg)
print('const window losses:', [f'{l:.2e}' for l in pinn_const['win_losses']])
pinn_dyn = tr.solve_pinn('dyn', gamma1=1.0, checkpoint_dir=f"{CKPT_DIR}/dyn", **cfg)
print('dyn window losses:', [f'{l:.2e}' for l in pinn_dyn['win_losses']])

In [ ]:
# Overlay the reference only when its time span matches what is plotted.
figs = tr.plot_all(pinn_const, pinn_dyn,
                   ref_const=ref_const,
                   ref_dyn=ref_dyn if 'ref_dyn' in globals() else None,
                   show=True)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


# ============================================================
# Hill function
# ============================================================
def hill(x, K, n=1):
    x = max(x, 0.0)
    return x**n / (K**n + x**n)


# ============================================================
# Periodic dietary RA input + ATRA treatment
# ============================================================
def ra_input(tau, p):
    dietary = p["AR"] * (
        1.0 + np.cos((2.0 * np.pi * tau / p["TR"]) - p["phi"])
    )

    treatment = 0.5 * p["DR"] * (
        np.tanh(p["q"] * (tau - p["tau1"]))
        - np.tanh(p["q"] * (tau - p["tau2"]))
    )

    return p["mu0"] + dietary + treatment


# ============================================================
# Final nondimensional 7-variable model
# y = [b, p, h5, h13, m, r, c]
# ============================================================
def model(tau, y, p):
    b, apc, h5, h13, m, r, c = np.maximum(y, 0.0)

    W = p["W"]
    thetaP = p["thetaP"]

    deltaP = 1.0 + p["deltaP1"] * (1.0 - thetaP)
    muR = ra_input(tau, p)

    db = (
        W
        + p["eta13"] * hill(h13, p["kappa13"], p["nH"])
        - b
        - p["lambdaP"] * apc * b
        - p["lambda5"] * h5 * b / (p["kappa5"] + b)
    )

    dapc = (1.0 / p["epsP"]) * (
        (1.0 + p["rho5"] * h5)
        / (1.0 + p["rhoB"] * b + p["rho13"] * h13)
        - deltaP * apc
    )

    dh5 = (1.0 / p["eps5"]) * (
        p["a5"]
        + p["etaR"] * hill(r, p["kappaR"], 1)
        - h5
        - p["etaM"] * m * h5 / (p["kappaM"] + m)
    )

    dh13 = (1.0 / p["eps13"]) * (
        p["a13"]
        + p["etaB13"] * hill(b, p["kappaB13"], p["nB"])
        + p["etaM13"] * hill(m, p["kappaM13"], p["nM"])
        - h13
    )

    dm = (1.0 / p["epsM"]) * (
        p["aM"]
        + p["etaBM"] * hill(b, p["kappaBM"], p["nB"])
        - m
    )

    dr = (1.0 / p["epsR"]) * (
        muR
        - r
        - p["lambdaC"] * c * r
    )

    dc = (1.0 / p["epsC"]) * (
        p["aC"]
        + p["etaRC"] * hill(r, p["kappaRC"], 1)
        + p["etaBC"] * hill(b, p["kappaBC"], p["nB"])
        - c
    )

    return [db, dapc, dh5, dh13, dm, dr, dc]


# ============================================================
# Stemness index
# ============================================================
def stemness(sol, p):
    b = sol.y[0]
    apc = sol.y[1]
    h5 = sol.y[2]
    h13 = sol.y[3]

    return b * (1.0 + p["alpha13"] * h13) / (
        1.0 + apc + p["alpha5"] * h5
    )


# ============================================================
# Baseline parameters
# ============================================================
params = {
    # Biological regime
    "W": 0.80,
    "thetaP": 1.00,

    # Hill coefficients
    "nB": 2,
    "nM": 2,
    "nH": 2,

    # beta-catenin equation
    "eta13": 0.75,
    "kappa13": 0.55,
    "lambdaP": 1.60,
    "lambda5": 1.30,
    "kappa5": 0.50,

    # APC equation
    "epsP": 1.00,
    "rho5": 1.10,
    "rhoB": 1.10,
    "rho13": 1.30,
    "deltaP1": 3.50,

    # HOXA5 equation
    "eps5": 1.20,
    "a5": 0.15,
    "etaR": 2.50,
    "kappaR": 0.40,
    "etaM": 2.50,
    "kappaM": 0.50,

    # HOXA13 equation
    "eps13": 1.00,
    "a13": 0.18,
    "etaB13": 0.95,
    "kappaB13": 0.50,
    "etaM13": 0.55,
    "kappaM13": 0.50,

    # MYC equation
    "epsM": 0.60,
    "aM": 0.18,
    "etaBM": 1.35,
    "kappaBM": 0.50,

    # RA equation
    "epsR": 0.40,
    "lambdaC": 0.85,

    # CYP26A1 equation
    "epsC": 0.80,
    "aC": 0.08,
    "etaRC": 1.50,
    "kappaRC": 0.50,
    "etaBC": 1.50,
    "kappaBC": 0.50,

    # RA input: background + dietary periodic input + treatment
    "mu0": 0.35,
    "AR": 0.04,
    # "AR": 0.08,
    "TR": 24.0,
    "phi": 0.0,

    # ATRA treatment window
    "DR": 1.50,
    "q": 0.30,
    "tau1": 40.0,
    "tau2": 80.0,

    # Stemness
    "alpha13": 1.00,
    "alpha5": 1.00,
}


# ============================================================
# Biological regimes
# ============================================================
regimes = {
    "Normal": {
        "W": 0.80,
        "thetaP": 1.00,
    },
    "Early adenoma": {
        "W": 1.00,
        "thetaP": 0.75,
    },
    "Cancer-like": {
        "W": 1.50,
        "thetaP": 0.50,
    },
    "Strong APC-mutant": {
        "W": 2.00,
        "thetaP": 0.25,
    },
}


# ============================================================
# Initial conditions
# ============================================================
y0 = [
    0.20,  # b: beta-catenin
    1.00,  # p: APC
    0.80,  # h5: HOXA5
    0.30,  # h13: HOXA13
    0.30,  # m: MYC
    0.60,  # r: RA
    0.40,  # c: CYP26A1
]


# ============================================================
# Time domain
# ============================================================
tau_span = (0.0, 150.0)
tau_eval = np.linspace(tau_span[0], tau_span[1], 3000)


# ============================================================
# Solve all regimes
# ============================================================
solutions = {}

for name, setting in regimes.items():
    pcopy = params.copy()
    pcopy.update(setting)

    sol = solve_ivp(
        fun=lambda tau, y: model(tau, y, pcopy),
        t_span=tau_span,
        y0=y0,
        t_eval=tau_eval,
        method="LSODA",
        rtol=1e-8,
        atol=1e-10,
    )

    if not sol.success:
        raise RuntimeError(f"Simulation failed for {name}: {sol.message}")

    solutions[name] = (sol, pcopy)


# ============================================================
# Plot model variables
# ============================================================
variables = [
    r"$\beta$-catenin",
    "APC",
    "HOXA5",
    "HOXA13",
    "MYC",
    "RA",
    "CYP26A1",
]

for i, var in enumerate(variables):
    plt.figure(figsize=(7.2, 4.2))

    for name, (sol, pcopy) in solutions.items():
        plt.plot(sol.t, sol.y[i], linewidth=2.0, label=name)

    plt.axvspan(
        params["tau1"],
        params["tau2"],
        alpha=0.15,
        label="ATRA treatment",
    )

    plt.xlabel(r"Nondimensional time $\tau$")
    plt.ylabel(var)
    plt.title(var)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()


# ============================================================
# Plot RA input
# ============================================================
plt.figure(figsize=(7.2, 4.2))
mu_vals = np.array([ra_input(t, params) for t in tau_eval])

plt.plot(tau_eval, mu_vals, linewidth=2.0)
plt.axvspan(params["tau1"], params["tau2"], alpha=0.15, label="ATRA treatment")
plt.xlabel(r"Nondimensional time $\tau$")
plt.ylabel(r"$\mu_R(\tau)$")
plt.title("Periodic dietary RA input plus ATRA treatment")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# Plot stemness
# ============================================================
plt.figure(figsize=(7.2, 4.2))

for name, (sol, pcopy) in solutions.items():
    S = stemness(sol, pcopy)
    plt.plot(sol.t, S, linewidth=2.0, label=name)

plt.axvspan(params["tau1"], params["tau2"], alpha=0.15, label="ATRA treatment")
plt.xlabel(r"Nondimensional time $\tau$")
plt.ylabel("Stemness index")
plt.title("Stemness index across biological regimes")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


# ============================================================
# Print final and treatment-window summary
# ============================================================
print("\nSummary values")
print("-" * 90)

for name, (sol, pcopy) in solutions.items():
    S = stemness(sol, pcopy)

    final = sol.y[:, -1]
    final_S = S[-1]

    treatment_mask = (sol.t >= params["tau1"]) & (sol.t <= params["tau2"])
    min_S_treatment = np.min(S[treatment_mask])
    mean_S_treatment = np.mean(S[treatment_mask])

    print(f"\n{name}")
    print(f"Final beta-catenin:      {final[0]:.4f}")
    print(f"Final APC:              {final[1]:.4f}")
    print(f"Final HOXA5:            {final[2]:.4f}")
    print(f"Final HOXA13:           {final[3]:.4f}")
    print(f"Final MYC:              {final[4]:.4f}")
    print(f"Final RA:               {final[5]:.4f}")
    print(f"Final CYP26A1:          {final[6]:.4f}")
    print(f"Final stemness:         {final_S:.4f}")
    print(f"Minimum S during ATRA:  {min_S_treatment:.4f}")
    print(f"Mean S during ATRA:     {mean_S_treatment:.4f}")

ModuleNotFoundError: No module named 'numpy'